### День 1 — Базовая модель банковских счетов (усложнённый вариант)

🎯 Цель дня
Создать расширенную и абстрактную модель банковского счёта, способную служить базой для более сложных типов счетов.

📋 Требования

1. Абстрактный класс AbstractAccount
Создать абстрактный класс, содержащий:
- 🔑 уникальный идентификатор счёта
- 👤 данные владельца
- 💰 защищённый баланс
- 📊 статус счёта: активный, замороженный, закрытый
- 🔧 абстрактные методы:
  deposit(amount)
  withdraw(amount)
  get_account_info()

In [1]:
from dataclasses import dataclass
from abc import ABC, abstractmethod

'''
@dataclass cтрока-инструкция для Python: перед тем как окончательно создать
класс ниже, пропусти его через функцию dataclass
'''

@dataclass 
class AbstractAccount(ABC):
    account_id: str
    owner: str
    _balance: float
    status: str

    @abstractmethod 
    def deposit(self, amount): #положить деньги на счёт
        ...
        
    @abstractmethod
    def withdraw(self, amount): #снять деньги со счёта
        ...
        
    @abstractmethod
    def get_account_info(self): #показать выписку по счёту
        ...    

2. Класс BankAccount
Реализовать конкретный тип счёта с расширенными возможностями:
- ✅ валидация входящих данных
- 🔒 логические статусы и запрет операций при неверных статусах
- 🆔 автоматическая генерация короткого UUID при отсутствии номера счёта
- 💱 атрибут currency: RUB, USD, EUR, KZT, CNY

3. Исключения
Создать собственные классы ошибок:
- ❄️ AccountFrozenError счёт временно заблокирован
- 🚫 AccountClosedError счёта больше не существует
- ⚠️ InvalidOperationError то, что вы просите сделать, некорректно в принципе
- 💸 InsufficientFundsError денег не хватает

4. Базовые операции
Добавить проверки:
- ✅ корректность суммы
- 🔄 проверка статуса счета
- 🛡️ защита от отрицательных значений

5. Строковое представление
Метод __str__ должен показывать:
- 🏦 тип счета
- 👤 клиента
- 🔢 последние 4 цифры номера
- 📊 статус
- 💰 баланс и валюту



In [2]:
'''__init__ возьми параметры, которые пришли в BankAccount.__init__, 
и передай их дальше — пусть AbstractAccount.__init__ сделает свою обычную работу 
(сохранит их в self.account_id, self.owner и т.д.
'''
#исключения
class AccountFrozenError(Exception):
    '''Cчёт временно заблокирован'''
    pass

class AccountClosedError(Exception):
    '''Cчёт больше не существует'''
    pass

class InvalidOperationError(Exception):
    '''То, что вы просите сделать, некорректно в принципе'''
    pass

class InsufficientFundsError(Exception):
    '''Недостаточно средств на счёте'''
    pass

class TemporaryOperationError(Exception):
    """Временная ошибка — операция сейчас невозможна, но может пройти позже (например, ночное ограничение)"""
    pass

import uuid

class BankAccount(AbstractAccount):
    def __init__(self, owner, _balance, status, account_id=None, currency="RUB"):
        #валидация входящих данных
        if not owner:
            raise InvalidOperationError("Владелец счёта не может быть пустым")
        if not isinstance(_balance, (int, float)):
            raise InvalidOperationError(f"Баланс должен быть числом, получено: {type(_balance).__name__}")
        if _balance < 0:
            raise InvalidOperationError("Баланс не может быть отрицательным")
        if status not in ["active", "frozen", "closed"]:
            raise InvalidOperationError(f"Недопустимый статус: {status}")
        #автоматическая генерация короткого UUID при отсутствии номера счёта
        if account_id is None:
            account_id = uuid.uuid4().hex[:8]
        else:
            account_id = str(account_id)
            if not account_id:
                raise InvalidOperationError("Номер счёта не может быть пустым")
        if currency not in ["RUB", "USD", "EUR", "KZT", "CNY"]:
            raise InvalidOperationError(f"Недопустимая валюта: {currency}")
        super().__init__(account_id, owner, _balance, status)
        self.currency = currency

    def deposit(self, amount): #положить деньги на счёт
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(
                f"Операция разрешена только для активного счёта, текущий статус: {self.status}"
            )
        self._balance += amount   

    def withdraw(self, amount): #снять деньги со счёта
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(
                f"Операция разрешена только для активного счёта, текущий статус: {self.status}"
            )
        if self._balance - amount < 0:
            raise InsufficientFundsError("Недостаточно средств на счёте")
        self._balance -= amount

    def get_account_info(self):
        return {
            "owner": self.owner,
            "balance": self._balance,
            "status": self.status,
        }

    def __str__(self):
        return f"Тип счёта {type(self).__name__}, клиент {self.owner}, счёт ...{self.account_id[-4:]}, статус {self.status}, баланс {self._balance} {self.currency}"

In [3]:
acc = BankAccount("Иван", 1000.0, "active", currency="USD")
print(acc)

Тип счёта BankAccount, клиент Иван, счёт ...f010, статус active, баланс 1000.0 USD


6. Тестирование
Создать демонстрацию:
- ➕ создание активного и замороженного счёта
- 🚫 попытка операций над замороженным счётом
- ✅ валидное пополнение и снятие

In [4]:
print("\n=== 1. Создание активного счёта ===")
active_acc = BankAccount("Иван", 1000.0, "active", currency="RUB")
print(active_acc)


=== 1. Создание активного счёта ===
Тип счёта BankAccount, клиент Иван, счёт ...624b, статус active, баланс 1000.0 RUB


In [5]:
print("\n=== 2. Создание замороженного счёта ===")
frozen_acc = BankAccount("Мария", 500.0, "frozen", currency="USD")
print(frozen_acc)


=== 2. Создание замороженного счёта ===
Тип счёта BankAccount, клиент Мария, счёт ...6b1d, статус frozen, баланс 500.0 USD


In [6]:
print("\n=== 3. Попытка пополнить замороженный счёт ===")
try:
    frozen_acc.deposit(100)
except AccountFrozenError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 3. Попытка пополнить замороженный счёт ===
Ошибка (ожидаемо): Счёт заморожен, операция запрещена


In [7]:
print("\n=== 4. Попытка снять деньги с того же замороженного счёта ===") 
try:
    frozen_acc.withdraw(50)
except AccountFrozenError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 4. Попытка снять деньги с того же замороженного счёта ===
Ошибка (ожидаемо): Счёт заморожен, операция запрещена


In [8]:
print("\n=== 5. Попытка пополнить счёт со статусом blocked ===") 
blocked_acc = BankAccount("Иван", 1000.0, "active", currency="RUB")
blocked_acc.status = "blocked"
try:
    blocked_acc.deposit(100)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 5. Попытка пополнить счёт со статусом blocked ===
Ошибка (ожидаемо): Операция разрешена только для активного счёта, текущий статус: blocked


In [9]:
print("\n=== 6. Попытка снять деньги с того же счёта со статусом blocked ===") 
try:
    blocked_acc.withdraw(50)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 6. Попытка снять деньги с того же счёта со статусом blocked ===
Ошибка (ожидаемо): Операция разрешена только для активного счёта, текущий статус: blocked


In [10]:
print("\n=== 7. Валидное пополнение счёта ===") 
active_acc.deposit(500)
print(active_acc)


=== 7. Валидное пополнение счёта ===
Тип счёта BankAccount, клиент Иван, счёт ...624b, статус active, баланс 1500.0 RUB


In [11]:
print("\n=== 8. Валидное снятие счёта ===") 
active_acc.withdraw(200)
print(active_acc)


=== 8. Валидное снятие счёта ===
Тип счёта BankAccount, клиент Иван, счёт ...624b, статус active, баланс 1300.0 RUB


In [12]:
print("\n=== 9. Номер передан числом — приводится к строке, print не падает ===") 
acc_num = BankAccount("Иван", 1000.0, "active", account_id=12345)
print(acc_num)


=== 9. Номер передан числом — приводится к строке, print не падает ===
Тип счёта BankAccount, клиент Иван, счёт ...2345, статус active, баланс 1000.0 RUB


### День 2 — Базовая модель банковских счетов (усложнённый вариант)
🎯 Цель дня
Реализовать несколько дочерних классов счетов с расширенными возможностями.

📋 Требования

1. Наследование
Создать классы:
- SavingsAccount
- PremiumAccount
- InvestmentAccount

2. Функционал SavingsAccount
- 🔒 min_balance — минимальный остаток
- 📈 месячная ставка доходности
- 💰 метод apply_monthly_interest()

3. Функционал PremiumAccount
- ⬆️ увеличенные лимиты
- 💳 возможность овердрафта
- 📊 фиксированная комиссия

4. Функционал InvestmentAccount
- 📊 инвестиционные портфели
- 💼 виртуальные активы (stocks, bonds, etf)
- 📈 метод project_yearly_growth()

5. Полиморфизм
Каждый тип должен переопределять:
- withdraw()
- get_account_info()
- __str__()


In [13]:
class SavingsAccount(BankAccount):
    def __init__(self, owner, _balance, status, min_balance, interest_rate, account_id=None, currency="RUB"):
        super().__init__(owner, _balance, status, account_id, currency)
        if not isinstance(min_balance, (int, float)):
            raise InvalidOperationError(f"min_balance должен быть числом, получено: {type(min_balance).__name__}")
        if min_balance < 0:
            raise InvalidOperationError("min_balance не может быть отрицательным")
        if not isinstance(interest_rate, (int, float)):
            raise InvalidOperationError(f"interest_rate должен быть числом, получено: {type(interest_rate).__name__}")
        if interest_rate < 0:
            raise InvalidOperationError("interest_rate не может быть отрицательным")
        if _balance < min_balance:
            raise InvalidOperationError(
                f"Начальный баланс {_balance} не может быть меньше минимального остатка {min_balance}"
            )
        self.min_balance = min_balance
        self.interest_rate = interest_rate

    def apply_monthly_interest(self):
        if self.status != "active":
            raise InvalidOperationError(f"Операция разрешена только для активного счёта, текущий статус: {self.status}")
        interest = self._balance * self.interest_rate
        self._balance += interest
        return interest

    def withdraw(self, amount):
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(f"Операция разрешена только для активного счёта, текущий статус: {self.status}")
        if self._balance - amount < self.min_balance:
            raise InsufficientFundsError(f"Нельзя снять: баланс станет ниже минимального остатка {self.min_balance}")
        super().withdraw(amount)

    def get_account_info(self):
        info = super().get_account_info()
        info["min_balance"] = self.min_balance
        info["interest_rate"] = self.interest_rate
        return info

    def __str__(self):
        base = super().__str__()
        return f"{base}, мин. остаток {self.min_balance}, ставка {self.interest_rate}"

In [14]:
class PremiumAccount(BankAccount):
    def __init__(self, owner, _balance, status, overdraft_limit, transaction_fee, account_id=None, currency="RUB"):
        super().__init__(owner, _balance, status, account_id, currency)
        if not isinstance(overdraft_limit, (int, float)):
            raise InvalidOperationError(f"overdraft_limit должен быть числом, получено: {type(overdraft_limit).__name__}")
        if overdraft_limit < 0:
            raise InvalidOperationError("overdraft_limit не может быть отрицательным")
        if not isinstance(transaction_fee, (int, float)):
            raise InvalidOperationError(f"transaction_fee должен быть числом, получено: {type(transaction_fee).__name__}")
        if transaction_fee < 0:
            raise InvalidOperationError("transaction_fee не может быть отрицательным")
        self.overdraft_limit = overdraft_limit
        self.transaction_fee = transaction_fee
    
    def withdraw(self, amount):
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(f"Операция разрешена только для активного счёта, текущий статус: {self.status}")
        if self._balance - amount - self.transaction_fee < -self.overdraft_limit:
            raise InsufficientFundsError("Превышен лимит овердрафта")
        self._balance -= (amount + self.transaction_fee)
   
    def get_account_info(self):
        info = super().get_account_info()
        info["overdraft_limit"] = self.overdraft_limit
        info["transaction_fee"] = self.transaction_fee
        return info

    def __str__(self):
        base = super().__str__()
        return f"{base}, овердрафт {self.overdraft_limit}, комиссия {self.transaction_fee}"

In [15]:
class InvestmentAccount(BankAccount):
    ASSET_GROWTH_RATES = {"stocks": 0.08, "bonds": 0.03, "etf": 0.05}

    def __init__(self, owner, _balance, status, portfolio=None, account_id=None, currency="RUB"):
        super().__init__(owner, _balance, status, account_id, currency)
        portfolio = portfolio if portfolio is not None else {}
        if not isinstance(portfolio, dict):
            raise InvalidOperationError(f"portfolio должен быть словарём, получено: {type(portfolio).__name__}")
        for asset_type, amount in portfolio.items():
            if asset_type not in self.ASSET_GROWTH_RATES:
                raise InvalidOperationError(
                    f"Недопустимый тип актива: {asset_type}. Разрешены: {list(self.ASSET_GROWTH_RATES.keys())}"
                )
            if not isinstance(amount, (int, float)):
                raise InvalidOperationError(f"Сумма актива '{asset_type}' должна быть числом, получено: {type(amount).__name__}")
            if amount < 0:
                raise InvalidOperationError(f"Сумма актива '{asset_type}' не может быть отрицательной")
        self.portfolio = portfolio

    def project_yearly_growth(self):
        total_growth = 0
        for asset_type, amount in self.portfolio.items():
            rate = self.ASSET_GROWTH_RATES.get(asset_type, 0)
            total_growth += amount * rate
        return total_growth

    def withdraw(self, amount):
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(f"Операция разрешена только для активного счёта, текущий статус: {self.status}")
        invested_total = sum(self.portfolio.values())
        free_balance = self._balance - invested_total
        if amount > free_balance:
            raise InsufficientFundsError(
                f"Нельзя снять: {invested_total} уже вложено в портфель, свободно только {free_balance}"
            )
        super().withdraw(amount)

    def get_account_info(self):
        info = super().get_account_info()
        info["portfolio"] = self.portfolio
        info["projected_yearly_growth"] = self.project_yearly_growth()
        return info

    def __str__(self):
        base = super().__str__()
        return f"{base}, портфель {self.portfolio}"



6. Тестирование
Создать несколько счетов каждого типа и выполнить различные операции.



In [16]:
print("=== 1. Создание счетов трёх типов ===")
savings = SavingsAccount("Иван", 1000.0, "active", min_balance=100.0, interest_rate=0.02)
premium = PremiumAccount("Мария", 1000.0, "active", overdraft_limit=500.0, transaction_fee=10.0)
invest = InvestmentAccount("Пётр", 10000.0, "active", portfolio={"stocks": 5000, "bonds": 3000, "etf": 2000})
print(savings)
print(premium)
print(invest)

=== 1. Создание счетов трёх типов ===
Тип счёта SavingsAccount, клиент Иван, счёт ...0b25, статус active, баланс 1000.0 RUB, мин. остаток 100.0, ставка 0.02
Тип счёта PremiumAccount, клиент Мария, счёт ...cdf2, статус active, баланс 1000.0 RUB, овердрафт 500.0, комиссия 10.0
Тип счёта InvestmentAccount, клиент Пётр, счёт ...11c3, статус active, баланс 10000.0 RUB, портфель {'stocks': 5000, 'bonds': 3000, 'etf': 2000}


In [17]:
print("\n=== 2. SavingsAccount: начисление процентов и снятие с защитой min_balance ===")
savings.apply_monthly_interest()
print(f"После начисления процентов: {savings}")
try:
    savings.withdraw(950)
except InsufficientFundsError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 2. SavingsAccount: начисление процентов и снятие с защитой min_balance ===
После начисления процентов: Тип счёта SavingsAccount, клиент Иван, счёт ...0b25, статус active, баланс 1020.0 RUB, мин. остаток 100.0, ставка 0.02
Ошибка (ожидаемо): Нельзя снять: баланс станет ниже минимального остатка 100.0


In [18]:
print("\n=== 3. PremiumAccount: снятие с уходом в овердрафт ===")
premium.withdraw(1300)
print(f"После снятия с овердрафтом: {premium}")


=== 3. PremiumAccount: снятие с уходом в овердрафт ===
После снятия с овердрафтом: Тип счёта PremiumAccount, клиент Мария, счёт ...cdf2, статус active, баланс -310.0 RUB, овердрафт 500.0, комиссия 10.0


In [19]:
print("\n=== 4. InvestmentAccount: прогноз роста и защита вложенных средств ===")
print(f"Прогноз роста портфеля за год: {invest.project_yearly_growth()}")

try:
    invest.withdraw(9000)  # вложено 10000, свободно 0 — должна быть ошибка
except InsufficientFundsError as e:
    print(f"Ошибка (ожидаемо): {e}")




=== 4. InvestmentAccount: прогноз роста и защита вложенных средств ===
Прогноз роста портфеля за год: 590.0
Ошибка (ожидаемо): Нельзя снять: 10000 уже вложено в портфель, свободно только 0.0


In [20]:
print("\n=== 5. InvestmentAccount: валидное снятие свободного остатка ===")
invest2 = InvestmentAccount("Анна", 12000.0, "active", portfolio={"stocks": 5000, "bonds": 3000, "etf": 2000})
print(f"Свободный остаток: {invest2._balance - sum(invest2.portfolio.values())}")
invest2.withdraw(1000)  # вложено 10000, свободно 2000 → 1000 можно снять
print(f"После снятия: {invest2}")


=== 5. InvestmentAccount: валидное снятие свободного остатка ===
Свободный остаток: 2000.0
После снятия: Тип счёта InvestmentAccount, клиент Анна, счёт ...81b0, статус active, баланс 11000.0 RUB, портфель {'stocks': 5000, 'bonds': 3000, 'etf': 2000}


In [21]:
print("\n=== 6. Тест: начальный баланс ниже min_balance (замечание 2) ===")
try:
    bad_savings = SavingsAccount("Света", 50.0, "active", min_balance=100.0, interest_rate=0.02)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 6. Тест: начальный баланс ниже min_balance (замечание 2) ===
Ошибка (ожидаемо): Начальный баланс 50.0 не может быть меньше минимального остатка 100.0


In [22]:
print("\n=== 7. Тест: недопустимый тип актива в портфеле (замечание 3) ===")
try:
    bad_portfolio = InvestmentAccount("Ксения", 5000.0, "active", portfolio={"crypto": 1000})
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

print("\n=== 8. Тест: нечисловая сумма актива в портфеле (замечание 3) ===")
try:
    bad_portfolio2 = InvestmentAccount("Ксения", 5000.0, "active", portfolio={"stocks": "много"})
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

print("\n=== 9. Тест: отрицательная сумма актива в портфеле (замечание 3) ===")
try:
    bad_portfolio3 = InvestmentAccount("Ксения", 5000.0, "active", portfolio={"stocks": -500})
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 7. Тест: недопустимый тип актива в портфеле (замечание 3) ===
Ошибка (ожидаемо): Недопустимый тип актива: crypto. Разрешены: ['stocks', 'bonds', 'etf']

=== 8. Тест: нечисловая сумма актива в портфеле (замечание 3) ===
Ошибка (ожидаемо): Сумма актива 'stocks' должна быть числом, получено: str

=== 9. Тест: отрицательная сумма актива в портфеле (замечание 3) ===
Ошибка (ожидаемо): Сумма актива 'stocks' не может быть отрицательной


In [23]:
print("\n=== 10. Тест: нечисловая сумма в withdraw дочерних классов (замечание 1) ===")
for acc, name in [(savings, "SavingsAccount"), (premium, "PremiumAccount"), (invest, "InvestmentAccount")]:
    try:
        acc.withdraw("сто")
    except InvalidOperationError as e:
        print(f"{name}: Ошибка (ожидаемо): {e}")


=== 10. Тест: нечисловая сумма в withdraw дочерних классов (замечание 1) ===
SavingsAccount: Ошибка (ожидаемо): Сумма должна быть числом, получено: str
PremiumAccount: Ошибка (ожидаемо): Сумма должна быть числом, получено: str
InvestmentAccount: Ошибка (ожидаемо): Сумма должна быть числом, получено: str


### День 3 — Система Bank

🎯 Цель дня
Реализовать управляющий класс банка с поддержкой клиентов и безопасности.

📋 Требования

1. Класс Client
- 👤 ФИО, ID, статус
- 🔢 список номеров счетов
- 📞 контакты
- ✅ проверка возраста >= 18

2. Класс Bank
Методы:
- add_client()
- open_account()
- close_account()
- freeze_account()
- unfreeze_account()
- authenticate_client()
- search_accounts()

3. Защита
- 🔒 3 неверные попытки входа = блокировка
- ⚠️ пометка подозрительных действий
- 🌙 запрет операций с 00:00 до 05:00

4. Дополнительно
- get_total_balance()
- get_clients_ranking()

5. Тестирование
Создание нескольких клиентов, открытие счетов, попытки входа, заморозка счетов.

In [24]:
class Client:
    def __init__(self, full_name, client_id, age, phone, status="active"):
        if not isinstance(full_name, str):
            raise InvalidOperationError(f"ФИО должно быть строкой, получено: {type(full_name).__name__}")
        if not full_name:
            raise InvalidOperationError("ФИО не может быть пустым")
        if not isinstance(client_id, str):
            raise InvalidOperationError(f"ID клиента должен быть строкой, получено: {type(client_id).__name__}")
        if not client_id:
            raise InvalidOperationError("ID клиента не может быть пустым")
        if not isinstance(age, (int, float)):
            raise InvalidOperationError(f"Возраст должен быть числом, получено: {type(age).__name__}")
        if age < 18:
            raise InvalidOperationError("Клиент должен быть совершеннолетним (18+)")
        if not isinstance(phone, str):
            raise InvalidOperationError(f"Телефон должен быть строкой, получено: {type(phone).__name__}")
        if not phone:
            raise InvalidOperationError("Телефон не может быть пустым")
        if status not in ["active", "blocked"]:
            raise InvalidOperationError(f"Недопустимый статус клиента: {status}")

        self.full_name = full_name
        self.client_id = client_id
        self.age = age
        self.phone = phone
        self.status = status
        self.account_ids = []

In [25]:
client1 = Client("Иванов Иван Иванович", "C001", 25, "+79991234567")
print(client1.full_name, client1.account_ids)

try:
    client2 = Client("Петров Пётр", "C002", "25", "+79997654321")  # возраст строкой
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

Иванов Иван Иванович []
Ошибка (ожидаемо): Возраст должен быть числом, получено: str


In [26]:
from datetime import datetime

class Bank:
    def __init__(self, name):
        self.name = name
        self.clients = {}   # client_id -> объект Client
        self.accounts = {}  # account_id -> объект BankAccount (или наследника)
        self.passwords = {}         # client_id -> пароль
        self.failed_attempts = {}   # client_id -> счётчик неверных попыток
        self.suspicious_activity_log = []   # список записей о подозрительных действиях
        self.closed_account_attempts = {}   # account_id -> счётчик попыток тронуть frozen/closed счёт
        self.night_attempts = {}   # client_id -> счётчик попыток операций ночью

    def add_client(self, client, password):
        if not isinstance(client, Client):
            raise InvalidOperationError(f"Ожидался объект Client, получено: {type(client).__name__}")
        if not isinstance(password, str) or not password:
            raise InvalidOperationError("Пароль должен быть непустой строкой")
        if client.client_id in self.clients:
            raise InvalidOperationError(f"Клиент с ID {client.client_id} уже существует")
        self.clients[client.client_id] = client
        self.passwords[client.client_id] = password
        self.failed_attempts[client.client_id] = 0

    def open_account(self, client_id, account_type="basic", **kwargs):
        self._check_operating_hours(client_id)
        
        if not isinstance(client_id, str):
            raise InvalidOperationError(f"client_id должен быть строкой, получено: {type(client_id).__name__}")
    
        if client_id not in self.clients:
            raise InvalidOperationError(f"Клиент с ID {client_id} не найден")
        client = self.clients[client_id]
    
        if client.status == "blocked":
            raise InvalidOperationError("Клиент заблокирован, операция запрещена")
    
        account_classes = {
            "basic": BankAccount,
            "savings": SavingsAccount,
            "premium": PremiumAccount,
            "investment": InvestmentAccount,
        }
        if account_type not in account_classes:
            raise InvalidOperationError(f"Недопустимый тип счёта: {account_type}. Разрешены: {list(account_classes.keys())}")
    
        account_class = account_classes[account_type]
        try:
            account = account_class(owner=client.full_name, **kwargs)
        except TypeError as e:
            raise InvalidOperationError(f"Некорректные параметры для счёта типа '{account_type}': {e}")
    
        if account.account_id in self.accounts:
            raise InvalidOperationError(f"Счёт с ID {account.account_id} уже существует в системе")
    
        self.accounts[account.account_id] = account
        client.account_ids.append(account.account_id)
        return account

    def freeze_account(self, account_id):
        self._check_operating_hours(self._find_client_id_by_account(account_id))
        
        if not isinstance(account_id, str):
            raise InvalidOperationError(f"account_id должен быть строкой, получено: {type(account_id).__name__}")
        if account_id not in self.accounts:
            raise InvalidOperationError(f"Счёт с ID {account_id} не найден")
        account = self.accounts[account_id]
        if account.status == "closed":
            self.closed_account_attempts[account_id] = self.closed_account_attempts.get(account_id, 0) + 1
            if self.closed_account_attempts[account_id] == 2:
                self.suspicious_activity_log.append({
                    "type": "repeated_inactive_account_attempt",
                    "account_id": account_id,
                    "timestamp": datetime.now(),
                    "details": "Повторная попытка операции с неактивным счётом (freeze_account)"
                })
            raise AccountClosedError("Нельзя заморозить закрытый счёт")
        account.status = "frozen"
    
    def unfreeze_account(self, account_id):
        self._check_operating_hours(self._find_client_id_by_account(account_id))
        
        if not isinstance(account_id, str):
            raise InvalidOperationError(f"account_id должен быть строкой, получено: {type(account_id).__name__}")
    
        if account_id not in self.accounts:
            raise InvalidOperationError(f"Счёт с ID {account_id} не найден")
        account = self.accounts[account_id]
        
        if account.status == "closed":
            self.closed_account_attempts[account_id] = self.closed_account_attempts.get(account_id, 0) + 1
            if self.closed_account_attempts[account_id] == 2:
                self.suspicious_activity_log.append({
                    "type": "repeated_inactive_account_attempt",
                    "account_id": account_id,
                    "timestamp": datetime.now(),
                    "details": "Повторная попытка операции с неактивным счётом (unfreeze_account)"
                })
            raise AccountClosedError("Нельзя разморозить закрытый счёт")
        account.status = "active"
    
    def close_account(self, account_id):
        self._check_operating_hours(self._find_client_id_by_account(account_id))
        
        if not isinstance(account_id, str):
            raise InvalidOperationError(f"account_id должен быть строкой, получено: {type(account_id).__name__}")
    
        if account_id not in self.accounts:
            raise InvalidOperationError(f"Счёт с ID {account_id} не найден")
        account = self.accounts[account_id]
    
        if account.status == "closed":
            self.closed_account_attempts[account_id] = self.closed_account_attempts.get(account_id, 0) + 1
            if self.closed_account_attempts[account_id] == 2:
                self.suspicious_activity_log.append({
                    "type": "repeated_inactive_account_attempt",
                    "account_id": account_id,
                    "timestamp": datetime.now(),
                    "details": "Повторная попытка операции с неактивным счётом (close_account)"
                })
            raise AccountClosedError("Счёт уже закрыт")
        
        if account._balance != 0:
            raise InvalidOperationError("Нельзя закрыть счёт с ненулевым балансом")
        account.status = "closed"

    def authenticate_client(self, client_id, password):
        self._check_operating_hours(client_id)
        
        if not isinstance(client_id, str):
            raise InvalidOperationError(f"client_id должен быть строкой, получено: {type(client_id).__name__}")
        if client_id not in self.clients:
            raise InvalidOperationError(f"Клиент с ID {client_id} не найден")
    
        client = self.clients[client_id]
        if client.status == "blocked":
            raise InvalidOperationError("Клиент заблокирован из-за превышения попыток входа")
    
        if self.passwords[client_id] == password:
            self.failed_attempts[client_id] = 0
            return True
        else:
            self.failed_attempts[client_id] += 1
            if self.failed_attempts[client_id] == 2:
                self.suspicious_activity_log.append({
                    "type": "repeated_failed_login",
                    "client_id": client_id,
                    "timestamp": datetime.now(),
                    "details": "Вторая неудачная попытка входа подряд"
                })
            if self.failed_attempts[client_id] >= 3:
                client.status = "blocked"
                raise InvalidOperationError("Неверный пароль. Клиент заблокирован после 3 неудачных попыток")
            raise InvalidOperationError(
                f"Неверный пароль. Осталось попыток: {3 - self.failed_attempts[client_id]}"
            )
            
    def search_accounts(self, owner=None, status=None, account_type=None):
        results = []
        for account in self.accounts.values():
            if owner is not None and account.owner != owner:
                continue
            if status is not None and account.status != status:
                continue
            if account_type is not None and type(account).__name__ != account_type:
                continue
            results.append(account)
        return results

    def _check_operating_hours(self, client_id=None):
        current_hour = datetime.now().hour
        if 0 <= current_hour < 5:
            if client_id is not None:
                self.night_attempts[client_id] = self.night_attempts.get(client_id, 0) + 1
                if self.night_attempts[client_id] == 2:
                    self.suspicious_activity_log.append({
                        "type": "repeated_night_operation",
                        "client_id": client_id,
                        "timestamp": datetime.now(),
                        "details": "Повторная попытка операции в ночное время"
                    })
            raise TemporaryOperationError("Операции запрещены с 00:00 до 05:00")

    def _find_client_id_by_account(self, account_id):
        for client_id, client in self.clients.items():
            if account_id in client.account_ids:
                return client_id
        return None

    def get_total_balance(self):
        return sum(account._balance for account in self.accounts.values())

    def get_clients_ranking(self):
        ranking = []
        for client in self.clients.values():
            total = sum(self.accounts[acc_id]._balance for acc_id in client.account_ids)
            ranking.append((client, total))
        ranking.sort(key=lambda item: item[1], reverse=True)
        return ranking

In [27]:
print("=== 1. Создание нескольких клиентов и открытие счетов ===")
bank = Bank("МойБанк")

client1 = Client("Иванов Иван Иванович", "C001", 25, "+79991234567")
client2 = Client("Петрова Мария Сергеевна", "C002", 30, "+79997654321")
bank.add_client(client1, "secret123")
bank.add_client(client2, "secret456")

acc1 = bank.open_account("C001", account_type="basic", _balance=1000.0, status="active")
acc2 = bank.open_account("C001", account_type="savings", _balance=500.0, status="active", min_balance=100.0, interest_rate=0.02)
acc3 = bank.open_account("C002", account_type="basic", _balance=5000.0, status="active")

print(acc1)
print(acc2)
print(acc3)

=== 1. Создание нескольких клиентов и открытие счетов ===
Тип счёта BankAccount, клиент Иванов Иван Иванович, счёт ...9b25, статус active, баланс 1000.0 RUB
Тип счёта SavingsAccount, клиент Иванов Иван Иванович, счёт ...86e4, статус active, баланс 500.0 RUB, мин. остаток 100.0, ставка 0.02
Тип счёта BankAccount, клиент Петрова Мария Сергеевна, счёт ...4399, статус active, баланс 5000.0 RUB


In [28]:
print("\n=== 2. Аутентификация клиента ===")
result = bank.authenticate_client("C001", "secret123")
print(f"Успешный вход: {result}")

try:
    bank.authenticate_client("C002", "wrongpass")
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 2. Аутентификация клиента ===
Успешный вход: True
Ошибка (ожидаемо): Неверный пароль. Осталось попыток: 2


In [29]:
print("\n=== 3. Заморозка и разморозка счёта ===")
bank.freeze_account(acc1.account_id)
print(f"После заморозки: {acc1}")

bank.unfreeze_account(acc1.account_id)
print(f"После разморозки: {acc1}")




=== 3. Заморозка и разморозка счёта ===
После заморозки: Тип счёта BankAccount, клиент Иванов Иван Иванович, счёт ...9b25, статус frozen, баланс 1000.0 RUB
После разморозки: Тип счёта BankAccount, клиент Иванов Иван Иванович, счёт ...9b25, статус active, баланс 1000.0 RUB


In [30]:
print("\n=== 4. Поиск счетов ===")
found = bank.search_accounts(status="active")
print(f"Найдено активных счетов: {len(found)}")



=== 4. Поиск счетов ===
Найдено активных счетов: 3


In [31]:
print("\n=== 5. Общий баланс банка и рейтинг клиентов ===")
print(f"Общий баланс банка: {bank.get_total_balance()}")

ranking = bank.get_clients_ranking()
for client, total in ranking:
    print(f"{client.full_name}: {total}")


=== 5. Общий баланс банка и рейтинг клиентов ===
Общий баланс банка: 6500.0
Петрова Мария Сергеевна: 5000.0
Иванов Иван Иванович: 1500.0


In [32]:
print("\n=== 6. Механизм пометки подозрительных действий ===")
print("""
Это дополнительная защита сверх формальных требований задания.
Система логирует событие в bank.suspicious_activity_log, когда:
  1) клиент дважды подряд вводит неверный пароль (осталась 1 попытка до блокировки);
  2) клиент дважды и более пытается совершить операцию в запрещённое ночное время (00:00–05:00);
  3) кто-то дважды и более пытается выполнить операцию (freeze/unfreeze/close)
     над уже закрытым счётом — потенциальный признак использования устаревших данных.
Идея: одна ошибка — это случайность, а повторение — уже сигнал для проверки человеком.
""")

print("--- 6.1. Демонстрация: повторная неудачная попытка входа ---")
try:
    bank.authenticate_client("C002", "wrong1")
except InvalidOperationError as e:
    print(e)
try:
    bank.authenticate_client("C002", "wrong2")
except InvalidOperationError as e:
    print(e)

print("\n--- 6.2. Демонстрация: повторная попытка операции с закрытым счётом ---")
client_demo = Client("Демо Клиент", "C003_DEMO", 25, "+70000000099")
bank.add_client(client_demo, "demopass")
acc4 = bank.open_account("C003_DEMO", account_type="basic", _balance=0.0, status="active")
bank.close_account(acc4.account_id)
try:
    bank.freeze_account(acc4.account_id)
except AccountClosedError as e:
    print(e)
try:
    bank.close_account(acc4.account_id)
except AccountClosedError as e:
    print(e)

print("\n--- 6.3. Итоговый журнал подозрительной активности ---")
for entry in bank.suspicious_activity_log:
    print(entry)


=== 6. Механизм пометки подозрительных действий ===

Это дополнительная защита сверх формальных требований задания.
Система логирует событие в bank.suspicious_activity_log, когда:
  1) клиент дважды подряд вводит неверный пароль (осталась 1 попытка до блокировки);
  2) клиент дважды и более пытается совершить операцию в запрещённое ночное время (00:00–05:00);
  3) кто-то дважды и более пытается выполнить операцию (freeze/unfreeze/close)
     над уже закрытым счётом — потенциальный признак использования устаревших данных.
Идея: одна ошибка — это случайность, а повторение — уже сигнал для проверки человеком.

--- 6.1. Демонстрация: повторная неудачная попытка входа ---
Неверный пароль. Осталось попыток: 1
Неверный пароль. Клиент заблокирован после 3 неудачных попыток

--- 6.2. Демонстрация: повторная попытка операции с закрытым счётом ---
Нельзя заморозить закрытый счёт
Счёт уже закрыт

--- 6.3. Итоговый журнал подозрительной активности ---
{'type': 'repeated_failed_login', 'client_id':

In [33]:
print("\n=== 7. Исправление замечания: PremiumAccount.withdraw() запрещает операцию при статусе, отличном от active ===")
weird_premium = PremiumAccount("Тест", 1000.0, "active", overdraft_limit=500.0, transaction_fee=10.0)
weird_premium.status = "blocked"
try:
    weird_premium.withdraw(100)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 7. Исправление замечания: PremiumAccount.withdraw() запрещает операцию при статусе, отличном от active ===
Ошибка (ожидаемо): Операция разрешена только для активного счёта, текущий статус: blocked


In [34]:
print("\n=== 8. Исправление замечания: apply_monthly_interest() запрещён на неактивном счёте ===")
frozen_savings = SavingsAccount("Тест", 1000.0, "frozen", min_balance=100.0, interest_rate=0.02)
try:
    frozen_savings.apply_monthly_interest()
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 8. Исправление замечания: apply_monthly_interest() запрещён на неактивном счёте ===
Ошибка (ожидаемо): Операция разрешена только для активного счёта, текущий статус: frozen


In [35]:
print("\n=== 9. Исправление замечания: Bank.open_account() проверяет уникальность account_id ===")
bank_test = Bank("ТестБанк")
client_a = Client("Клиент А", "TA01", 25, "+70000000001")
client_b = Client("Клиент Б", "TB01", 30, "+70000000002")
bank_test.add_client(client_a, "pass1")
bank_test.add_client(client_b, "pass2")

bank_test.open_account("TA01", account_type="basic", _balance=1000.0, status="active", account_id="SAME001")
try:
    bank_test.open_account("TB01", account_type="basic", _balance=500.0, status="active", account_id="SAME001")
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 9. Исправление замечания: Bank.open_account() проверяет уникальность account_id ===
Ошибка (ожидаемо): Счёт с ID SAME001 уже существует в системе


In [36]:
print("\n=== 10. Исправление замечания: заблокированный клиент не может открыть счёт через open_account() ===")
client_c = Client("Клиент В", "TC01", 28, "+70000000003")
bank_test.add_client(client_c, "pass3")
client_c.status = "blocked"
try:
    bank_test.open_account("TC01", account_type="basic", _balance=1000.0, status="active")
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 10. Исправление замечания: заблокированный клиент не может открыть счёт через open_account() ===
Ошибка (ожидаемо): Клиент заблокирован, операция запрещена


### День 4 — Система транзакций

🎯 Цель дня
Создать расширенную модель транзакций и модуль обработки очередей.

📋 Требования

Transaction
- 🆔 ID, тип, сумма, валюта, комиссия
- 👤 отправитель, получатель
- 📊 статус, причина отказа
- ⏰ timestamps

TransactionQueue
- ➕ добавление
- ⭐ приоритет
- ⏳ отложенные операции
- ❌ отмена

TransactionProcessor
- 💰 комиссии
- 💱 конвертация
- 🔄 повторные попытки
- 📝 фиксация ошибок

Правила
- 🚫 запрет переводов при минусе (кроме премиум)
- 🔒 запрет на замороженные счета
- 💸 комиссия за внешние переводы

Тестирование
Создать 10 транзакций, поместить в очередь, выполнить

In [37]:
class Transaction:
    def __init__(self, sender_account_id, receiver_account_id, amount, currency,
                 transaction_type="transfer", commission=0.0, transaction_id=None):
        if not isinstance(sender_account_id, str) or not sender_account_id:
            raise InvalidOperationError("sender_account_id должен быть непустой строкой")
        if not isinstance(receiver_account_id, str) or not receiver_account_id:
            raise InvalidOperationError("receiver_account_id должен быть непустой строкой")
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"amount должен быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("amount должен быть положительным")
        if currency not in ["RUB", "USD", "EUR", "KZT", "CNY"]:
            raise InvalidOperationError(f"Недопустимая валюта: {currency}")
        if transaction_type not in ["transfer", "deposit", "withdrawal"]:
            raise InvalidOperationError(f"Недопустимый тип транзакции: {transaction_type}")
        if not isinstance(commission, (int, float)) or commission < 0:
            raise InvalidOperationError("commission должна быть неотрицательным числом")

        self.transaction_id = transaction_id or uuid.uuid4().hex[:8]
        self.sender_account_id = sender_account_id
        self.receiver_account_id = receiver_account_id
        self.amount = amount
        self.currency = currency
        self.transaction_type = transaction_type
        self.commission = commission
        self.status = "pending"
        self.failure_reason = None
        self.created_at = datetime.now()
        self.completed_at = None

    def __str__(self):
        return (f"Transaction {self.transaction_id}: {self.sender_account_id} → {self.receiver_account_id}, "
                f"{self.amount} {self.currency}, тип={self.transaction_type}, статус={self.status}")

In [38]:
class TransactionQueue:
    def __init__(self):
        self.queue = []   # список кортежей (priority, transaction)

    def add(self, transaction, priority=0, scheduled_at=None):
        if not isinstance(transaction, Transaction):
            raise InvalidOperationError(f"Ожидался объект Transaction, получено: {type(transaction).__name__}")
        if not isinstance(priority, (int, float)):
            raise InvalidOperationError(f"priority должен быть числом, получено: {type(priority).__name__}")
        if transaction.status not in ["pending", "failed"]:
            raise InvalidOperationError(f"Нельзя добавить в очередь транзакцию со статусом: {transaction.status}")
        if any(item["transaction"].transaction_id == transaction.transaction_id for item in self.queue):
            raise InvalidOperationError(f"Транзакция с ID {transaction.transaction_id} уже есть в очереди")

        self.queue.append({
            "transaction": transaction,
            "priority": priority,
            "scheduled_at": scheduled_at
        })

    def get_next(self):
        now = datetime.now()
        available = [
            item for item in self.queue
            if item["scheduled_at"] is None or item["scheduled_at"] <= now
        ]
        if not available:
            return None
        best_item = max(available, key=lambda item: item["priority"])
        self.queue.remove(best_item)
        return best_item["transaction"]

    def cancel(self, transaction_id):
        if not isinstance(transaction_id, str):
            raise InvalidOperationError(f"transaction_id должен быть строкой, получено: {type(transaction_id).__name__}")
        for item in self.queue:
            if item["transaction"].transaction_id == transaction_id:
                item["transaction"].status = "cancelled"
                self.queue.remove(item)
                return item["transaction"]
        raise InvalidOperationError(f"Транзакция с ID {transaction_id} не найдена в очереди")

In [39]:
import time

class TransactionProcessor:
    # Курсы валют статичны для целей учебного проекта.
    # В реальной системе курсы обновлялись бы регулярно (например, ежедневно из внешнего API).
    EXCHANGE_RATES_TO_RUB = {
        "RUB": 1.0,
        "USD": 90.0,
        "EUR": 100.0,
        "KZT": 0.2,
        "CNY": 12.5,
    }

    def __init__(self, bank):
        if not isinstance(bank, Bank):
            raise InvalidOperationError(f"Ожидался объект Bank, получено: {type(bank).__name__}")
        self.bank = bank

    def convert_currency(self, amount, from_currency, to_currency):
        if from_currency not in self.EXCHANGE_RATES_TO_RUB:
            raise InvalidOperationError(f"Неизвестная валюта: {from_currency}")
        if to_currency not in self.EXCHANGE_RATES_TO_RUB:
            raise InvalidOperationError(f"Неизвестная валюта: {to_currency}")
        amount_in_rub = amount * self.EXCHANGE_RATES_TO_RUB[from_currency]
        return round(amount_in_rub / self.EXCHANGE_RATES_TO_RUB[to_currency], 2)

    def process(self, transaction):
        if not isinstance(transaction, Transaction):
            raise InvalidOperationError(f"Ожидался объект Transaction, получено: {type(transaction).__name__}")
        if transaction.status not in ["pending", "failed"]:
            raise InvalidOperationError(f"Транзакция уже обработана, текущий статус: {transaction.status}")

        # ночное ограничение — единственная "временная" причина отказа в нашей системе
        self.bank._check_operating_hours()

        sender = self.bank.accounts.get(transaction.sender_account_id)
        if sender is None:
            transaction.status = "failed"
            transaction.failure_reason = "Счёт отправителя не найден"
            transaction.completed_at = datetime.now()
            return transaction

        if sender.status == "frozen":
            transaction.status = "failed"
            transaction.failure_reason = "Счёт отправителя заморожен"
            transaction.completed_at = datetime.now()
            return transaction
        if sender.status == "closed":
            transaction.status = "failed"
            transaction.failure_reason = "Счёт отправителя закрыт"
            transaction.completed_at = datetime.now()
            return transaction

        receiver = self.bank.accounts.get(transaction.receiver_account_id)
        is_external = receiver is None

        # проверяем получателя ДО списания денег у отправителя,
        # чтобы деньги не "терялись" при неудачном зачислении
        if not is_external and receiver.status != "active":
            transaction.status = "failed"
            transaction.failure_reason = "Счёт получателя недоступен для зачисления"
            transaction.completed_at = datetime.now()
            return transaction

        # правило: комиссия за внешние переводы
        commission = transaction.commission
        if is_external and commission == 0:
            commission = round(transaction.amount * 0.01, 2)

        total_debit = transaction.amount + commission

        # правило: запрет минуса, кроме PremiumAccount
        if not isinstance(sender, PremiumAccount):
            if sender._balance - total_debit < 0:
                transaction.status = "failed"
                transaction.failure_reason = "Недостаточно средств (уход в минус запрещён для этого типа счёта)"
                transaction.completed_at = datetime.now()
                return transaction

        try:
            sender.withdraw(total_debit)
        except (InsufficientFundsError, InvalidOperationError, AccountFrozenError, AccountClosedError) as e:
            transaction.status = "failed"
            transaction.failure_reason = str(e)
            transaction.completed_at = datetime.now()
            return transaction

        if not is_external:
            if receiver.currency != transaction.currency:
                converted_amount = self.convert_currency(transaction.amount, transaction.currency, receiver.currency)
            else:
                converted_amount = transaction.amount
            receiver.deposit(converted_amount)

        transaction.status = "completed"
        transaction.completed_at = datetime.now()
        return transaction

    def process_with_retry(self, transaction, max_retries=3, delay_seconds=2):
        attempt = 0
        while True:
            attempt += 1
            try:
                return self.process(transaction)
            except TemporaryOperationError as e:
                transaction.status = "failed"
                transaction.failure_reason = str(e)
                if attempt >= max_retries:
                    transaction.completed_at = datetime.now()
                    return transaction
                print(f"Попытка {attempt} не удалась ({e}), повтор через {delay_seconds} сек...")
                time.sleep(delay_seconds)

In [40]:
import types

bank_conv = Bank("БанкКонвертации")
client_conv = Client("Тестов Тест", "TC01", 25, "+70000000000")
bank_conv.add_client(client_conv, "pass123")
rub_acc = bank_conv.open_account("TC01", account_type="basic", _balance=1000.0, status="active", currency="RUB")
usd_acc = bank_conv.open_account("TC01", account_type="basic", _balance=100.0, status="active", currency="USD")

night_call_counter = {"count": 0}

def fake_check_operating_hours(self, client_id=None):
    night_call_counter["count"] += 1
    if night_call_counter["count"] < 3:
        raise TemporaryOperationError("Операции запрещены с 00:00 до 05:00")

bank_conv._check_operating_hours = types.MethodType(fake_check_operating_hours, bank_conv)

processor_conv2 = TransactionProcessor(bank_conv)
t_retry = Transaction(rub_acc.account_id, usd_acc.account_id, 100.0, "RUB")

result_retry = processor_conv2.process_with_retry(t_retry, max_retries=5, delay_seconds=1)
print(result_retry.status)
print(f"Всего вызовов проверки времени: {night_call_counter['count']}")

Попытка 1 не удалась (Операции запрещены с 00:00 до 05:00), повтор через 1 сек...
Попытка 2 не удалась (Операции запрещены с 00:00 до 05:00), повтор через 1 сек...
completed
Всего вызовов проверки времени: 3


In [41]:
queue_test = TransactionQueue()
t_dup = Transaction("acc001", "acc002", 100.0, "RUB")
queue_test.add(t_dup, priority=1)

try:
    queue_test.add(t_dup, priority=2)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

t_dup.status = "completed"
try:
    queue_test.add(t_dup, priority=1)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

Ошибка (ожидаемо): Транзакция с ID c652f181 уже есть в очереди
Ошибка (ожидаемо): Нельзя добавить в очередь транзакцию со статусом: completed


In [42]:
print("=== 1. Подготовка: банк, клиенты и счета разных типов ===")
bank4 = Bank("БанкТранзакций")

client_a = Client("Алексеев Алексей", "D001", 25, "+70000000101")
client_b = Client("Борисова Борислава", "D002", 30, "+70000000102")
bank4.add_client(client_a, "passA")
bank4.add_client(client_b, "passB")

basic_a = bank4.open_account("D001", account_type="basic", _balance=1000.0, status="active", currency="RUB")
basic_b = bank4.open_account("D002", account_type="basic", _balance=500.0, status="active", currency="RUB")
premium_a = bank4.open_account("D001", account_type="premium", _balance=200.0, status="active", overdraft_limit=500.0, transaction_fee=10.0)
usd_b = bank4.open_account("D002", account_type="basic", _balance=100.0, status="active", currency="USD")

processor4 = TransactionProcessor(bank4)
queue4 = TransactionQueue()

print(basic_a)
print(basic_b)
print(premium_a)
print(usd_b)

=== 1. Подготовка: банк, клиенты и счета разных типов ===
Тип счёта BankAccount, клиент Алексеев Алексей, счёт ...5717, статус active, баланс 1000.0 RUB
Тип счёта BankAccount, клиент Борисова Борислава, счёт ...ef54, статус active, баланс 500.0 RUB
Тип счёта PremiumAccount, клиент Алексеев Алексей, счёт ...ea11, статус active, баланс 200.0 RUB, овердрафт 500.0, комиссия 10.0
Тип счёта BankAccount, клиент Борисова Борислава, счёт ...8054, статус active, баланс 100.0 USD


In [43]:
print("\n=== 2. Создание 10 транзакций с разными сценариями ===")

t1 = Transaction(basic_a.account_id, basic_b.account_id, 200.0, "RUB")
t2 = Transaction(basic_b.account_id, basic_a.account_id, 100.0, "RUB")
t3 = Transaction(basic_a.account_id, "внешний_счёт_999", 150.0, "RUB")
t4 = Transaction(basic_a.account_id, usd_b.account_id, 90.0, "RUB")
t5 = Transaction(premium_a.account_id, basic_b.account_id, 500.0, "RUB")
t6 = Transaction(premium_a.account_id, "внешний_счёт_888", 300.0, "RUB")
t7 = Transaction(basic_b.account_id, basic_a.account_id, 10000.0, "RUB")
t8 = Transaction(basic_a.account_id, basic_b.account_id, 50.0, "RUB")
t9 = Transaction(basic_b.account_id, basic_a.account_id, 20.0, "RUB")
t10 = Transaction(usd_b.account_id, basic_a.account_id, 10.0, "USD")

queue4.add(t1, priority=1)
queue4.add(t2, priority=5)
queue4.add(t3, priority=1)
queue4.add(t4, priority=2)
queue4.add(t5, priority=1)
queue4.add(t6, priority=1)
queue4.add(t7, priority=1)
queue4.add(t8, priority=1)
queue4.add(t9, priority=3)
queue4.add(t10, priority=1)

print(f"Транзакций в очереди: {len(queue4.queue)}")


=== 2. Создание 10 транзакций с разными сценариями ===
Транзакций в очереди: 10


In [44]:
print("\n=== 3. Замораживаем basic_b перед обработкой, чтобы показать запрет операций ===")
bank4.freeze_account(basic_b.account_id)
print(basic_b)


=== 3. Замораживаем basic_b перед обработкой, чтобы показать запрет операций ===
Тип счёта BankAccount, клиент Борисова Борислава, счёт ...ef54, статус frozen, баланс 500.0 RUB


In [45]:
print("\n=== 4. Обработка всех транзакций из очереди (по приоритету) ===")
results = []
while True:
    transaction = queue4.get_next()
    if transaction is None:
        break
    result = processor4.process(transaction)
    results.append(result)
    print(f"{result.transaction_id}: {result.status} — {result.failure_reason or 'успешно'}")

print(f"\nВсего обработано транзакций: {len(results)}")


=== 4. Обработка всех транзакций из очереди (по приоритету) ===
4c062442: failed — Счёт отправителя заморожен
3b817b4f: failed — Счёт отправителя заморожен
549d29af: completed — успешно
65bb2d5b: failed — Счёт получателя недоступен для зачисления
523ff949: completed — успешно
c667d5ad: failed — Счёт получателя недоступен для зачисления
39deafc4: completed — успешно
8f2b9761: failed — Счёт отправителя заморожен
4c825a2d: failed — Счёт получателя недоступен для зачисления
b0fafeeb: completed — успешно

Всего обработано транзакций: 10


In [46]:
print("\n=== 5. Итоговые балансы счетов ===")
print(basic_a)
print(basic_b)
print(premium_a)
print(usd_b)

print("""
Пояснение 1: транзакция с premium_a на внешний счёт (t6) списывает две разные комиссии —
1% комиссию банка за внешний перевод и transaction_fee самого PremiumAccount.
Это осознанное решение: комиссия банка — плата за межбанковский перевод, комиссия
счёта — плата за обслуживание премиального счёта (овердрафт, лимиты). Это две
разные услуги, а не задвоение одной и той же комиссии.

Пояснение 2: basic_b был намеренно заморожен перед обработкой очереди, чтобы
продемонстрировать, что и как отправитель, и как получатель — заморозка счёта
блокирует любую транзакцию, связанную с ним, без списания средств.
""")


=== 5. Итоговые балансы счетов ===
Тип счёта BankAccount, клиент Алексеев Алексей, счёт ...5717, статус active, баланс 1658.5 RUB
Тип счёта BankAccount, клиент Борисова Борислава, счёт ...ef54, статус frozen, баланс 500.0 RUB
Тип счёта PremiumAccount, клиент Алексеев Алексей, счёт ...ea11, статус active, баланс -113.0 RUB, овердрафт 500.0, комиссия 10.0
Тип счёта BankAccount, клиент Борисова Борислава, счёт ...8054, статус active, баланс 91.0 USD

Пояснение 1: транзакция с premium_a на внешний счёт (t6) списывает две разные комиссии —
1% комиссию банка за внешний перевод и transaction_fee самого PremiumAccount.
Это осознанное решение: комиссия банка — плата за межбанковский перевод, комиссия
счёта — плата за обслуживание премиального счёта (овердрафт, лимиты). Это две
разные услуги, а не задвоение одной и той же комиссии.

Пояснение 2: basic_b был намеренно заморожен перед обработкой очереди, чтобы
продемонстрировать, что и как отправитель, и как получатель — заморозка счёта
блокирует